# 🚀 SmartStudyInstructor — Kaggle Wav2Lip Server

**Run Cell 1 FIRST** (setup), then **Cell 2** (server).

⚠️ Make sure you have **GPU enabled** in Kaggle settings.

In [ ]:
# ============================================================
# CELL 2 — DUAL T4 GPU LIPSYNC SERVER (Run AFTER Cell 1 finishes)
# ============================================================
import os, sys, base64, subprocess, traceback, threading, queue
import torch
from flask import Flask, request, jsonify
from PIL import Image

# ╔══════════════════════════════════════════════════════════╗
# ║  👉 PASTE YOUR NGROK TOKEN BELOW                        ║
# ╚══════════════════════════════════════════════════════════╝
NGROK_TOKEN = "3BZaGFS85TRNgCpNbFcjoTR0V5P_7KPvvJ52uQrtESVeEXAg9"

# ── Configuration ──
JOBS_DIR = "/kaggle/working/lipsync_jobs"
WAV2LIP_DIR = "/kaggle/working/Wav2Lip"
CHECKPOINT = os.path.join(WAV2LIP_DIR, "checkpoints", "wav2lip_gan.pth")

os.makedirs(JOBS_DIR, exist_ok=True)
os.makedirs(os.path.join(WAV2LIP_DIR, "temp"), exist_ok=True)

# Verify setup
assert os.path.exists(os.path.join(WAV2LIP_DIR, "inference.py")), \
    "❌ Wav2Lip/inference.py NOT FOUND! Run Cell 1 first!"
assert os.path.exists(CHECKPOINT), \
    "❌ Checkpoint NOT FOUND! Run Cell 1 first!"
print("✅ Wav2Lip verified")

# ── Multi-GPU Worker Pool Setup (T4 x 2) ──
gpu_queue = queue.Queue()
num_gpus = torch.cuda.device_count()
if num_gpus >= 2:
    gpu_queue.put("cuda:0")
    gpu_queue.put("cuda:1")
    print(f"🚀 DUAL GPU MODE ACTIVATED: 2x T4 GPUs ({torch.cuda.get_device_name(0)} & {torch.cuda.get_device_name(1)})")
elif num_gpus == 1:
    gpu_queue.put("cuda:0")
    print(f"⚡ SINGLE GPU MODE: {torch.cuda.get_device_name(0)}")
else:
    gpu_queue.put("cpu")
    print("⚠️ CPU MODE: No CUDA GPUs detected")

app = Flask(__name__)

def prepare_avatar(src, dst, size=720):
    """Resize avatar to square for Wav2Lip face detection."""
    img = Image.open(src).convert("RGB")
    w, h = img.size
    scale = min(size/w, size/h)
    nw, nh = int(w*scale), int(h*scale)
    img = img.resize((nw, nh), Image.LANCZOS)
    canvas = Image.new("RGB", (size, size), (0,0,0))
    canvas.paste(img, ((size-nw)//2, (size-nh)//2))
    canvas.save(dst, "PNG")
    print(f"  🖼️ Avatar: {w}x{h} → {size}x{size}")

import sys
sys.path.append(WAV2LIP_DIR)

print("\n🔧 Patching face_detection/api.py...")
# Patch face_detection/api.py to prevent TypeError on torch.device iteration
api_path = os.path.join(WAV2LIP_DIR, "face_detection", "api.py")
if os.path.exists(api_path):
    with open(api_path, "r") as f: api_code = f.read()
    if "if 'cuda' in device:" in api_code:
        api_code = api_code.replace("if 'cuda' in device:", "if 'cuda' in str(device):")
        with open(api_path, "w") as f: f.write(api_code)
        print("  ✅ Patched face_detection/api.py for torch.device string matching.")

def run_wav2lip(scene_id, audio_path, avatar_path, output_mp4, target_device="cuda:0"):
    """Run Wav2Lip inference on assigned GPU device with process isolation."""
    temp_dir = os.path.join(WAV2LIP_DIR, "temp")
    for f in os.listdir(temp_dir):
        try: os.remove(os.path.join(temp_dir, f))
        except: pass
    
    prep = os.path.join(JOBS_DIR, f"{scene_id}_prep.png")
    prepare_avatar(avatar_path, prep)
    
    print(f"  🚀 Running Wav2Lip on {target_device} for {scene_id}...")
    dev_str = str(target_device)
    gpu_id = str(dev_str.split(":")[-1]) if ":" in dev_str else "0"
    
    env = os.environ.copy()
    if torch.cuda.is_available() and ":" in dev_str:
        env["CUDA_VISIBLE_DEVICES"] = gpu_id
    
    cmd = [
        sys.executable, "inference.py",
        "--checkpoint_path", os.path.join(WAV2LIP_DIR, "checkpoints", "wav2lip_gan.pth"),
        "--face", prep,
        "--audio", audio_path,
        "--outfile", output_mp4,
        "--pads", "0", "10", "0", "0",
        "--resize_factor", "1",
        "--nosmooth",
        "--fps", "25.0",
        "--face_det_batch_size", "64",
        "--wav2lip_batch_size", "128"
    ]
    
    try:
        res = subprocess.run(
            cmd,
            cwd=WAV2LIP_DIR,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True
        )
        if res.returncode != 0:
            print(f"  ❌ Inference Exception on {target_device} (code {res.returncode}):\n{res.stdout}")
            return False
        else:
            for line in res.stdout.splitlines():
                if "100%" in line or "Load checkpoint" in line or "Model loaded" in line or "ffmpeg" in line:
                    print(f"    {line}")
    except Exception as e:
        print(f"  ❌ Inference Exception on {target_device}: {e}")
        traceback.print_exc()
        return False
    
    if os.path.exists(output_mp4) and os.path.getsize(output_mp4) > 1000:
        print(f"  ✅ Video completed on {target_device}: {os.path.getsize(output_mp4)} bytes")
        return True
    
    return False


@app.route("/health")
def health():
    gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else ["CPU"]
    return jsonify({"status": "online", "gpus": gpus, "gpu_count": len(gpus)})

@app.route("/clear_cache")
def clear_cache():
    import torch, gc
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return jsonify({"status": "cleared"})

@app.route("/generate_lipsync", methods=["POST"])
def generate_lipsync():
    try:
        data = request.json
        scene_id = data.get("scene_id", "scene_01")
        
        # Save audio
        audio_path = os.path.join(JOBS_DIR, f"{scene_id}.wav")
        with open(audio_path, "wb") as f:
            f.write(base64.b64decode(data["audio_base64"]))
        
        # Save avatar
        avatar_path = os.path.join(JOBS_DIR, f"{scene_id}_avatar.png")
        with open(avatar_path, "wb") as f:
            f.write(base64.b64decode(data["avatar_image_base64"]))
        
        output_mp4 = os.path.join(JOBS_DIR, f"{scene_id}_sync.mp4")
        
        # Acquire available GPU device from queue (cuda:0 or cuda:1)
        target_gpu = gpu_queue.get()
        print(f"🎬 {scene_id}: Acquired {target_gpu}")
        try:
            ok = run_wav2lip(scene_id, audio_path, avatar_path, output_mp4, target_device=target_gpu)
        finally:
            gpu_queue.put(target_gpu)
            print(f"  🔓 Released {target_gpu}")
        
        if not ok:
            raise Exception(f"Wav2Lip produced no output for {scene_id}")
        
        with open(output_mp4, "rb") as f:
            video_b64 = base64.b64encode(f.read()).decode()
        
        print(f"  ✅ {scene_id} complete on {target_gpu}!")
        return jsonify({"status": "success", "lipsync_video_base64": video_b64})
        
    except Exception as e:
        print(f"🔥 Error: {traceback.format_exc()}")
        return jsonify({"status": "error", "error_message": str(e)}), 500

# ── Start Server ──
from pyngrok import ngrok

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("NGROK_AUTH_TOKEN") or NGROK_TOKEN
except:
    token = NGROK_TOKEN

if not token or "paste_your" in token:
    print("❌ Paste your ngrok token in NGROK_TOKEN variable!")
else:
    ngrok.set_auth_token(token)
    try:
        for t in ngrok.get_tunnels(): ngrok.disconnect(t.public_url)
    except: pass
    tunnel = ngrok.connect(5000)
    url = tunnel.public_url
    if hasattr(tunnel, 'public_url'):
        url = tunnel.public_url
    else:
        url = str(tunnel)
    if 'https://' in url:
        import re
        match = re.search(r'(https://[^\s\"]+\.ngrok[^\s\"]*)', url)
        if match:
            url = match.group(1)
    print(f"\n🚀 PUBLIC URL: {url}")
    print(f"\n📋 Paste in backend/.env:")
    print(f"   CLOUD_RENDER_URL={url}")
    app.run(host="0.0.0.0", port=5000, threaded=True)

In [ ]:
# ============================================================
# CELL 2 — LIPSYNC SERVER (Run AFTER Cell 1 finishes)
# ============================================================
import os, sys, base64, subprocess, traceback, threading
from flask import Flask, request, jsonify
from PIL import Image

# ╔══════════════════════════════════════════════════════════╗
# ║  👉 PASTE YOUR NGROK TOKEN BELOW                        ║
# ╚══════════════════════════════════════════════════════════╝
NGROK_TOKEN = "3BZaGFS85TRNgCpNbFcjoTR0V5P_7KPvvJ52uQrtESVeEXAg9"

# ── Configuration ──
JOBS_DIR = "/kaggle/working/lipsync_jobs"
WAV2LIP_DIR = "/kaggle/working/Wav2Lip"
CHECKPOINT = os.path.join(WAV2LIP_DIR, "checkpoints", "wav2lip_gan.pth")

os.makedirs(JOBS_DIR, exist_ok=True)
os.makedirs(os.path.join(WAV2LIP_DIR, "temp"), exist_ok=True)

# Verify setup
assert os.path.exists(os.path.join(WAV2LIP_DIR, "inference.py")), \
    "❌ Wav2Lip/inference.py NOT FOUND! Run Cell 1 first!"
assert os.path.exists(CHECKPOINT), \
    "❌ Checkpoint NOT FOUND! Run Cell 1 first!"
print("✅ Wav2Lip verified")

# Only 1 scene at a time (prevents GPU race conditions)
inference_lock = threading.Lock()

app = Flask(__name__)

def prepare_avatar(src, dst, size=720):
    """Resize avatar to square for Wav2Lip face detection."""
    img = Image.open(src).convert("RGB")
    w, h = img.size
    scale = min(size/w, size/h)
    nw, nh = int(w*scale), int(h*scale)
    img = img.resize((nw, nh), Image.LANCZOS)
    canvas = Image.new("RGB", (size, size), (0,0,0))
    canvas.paste(img, ((size-nw)//2, (size-nh)//2))
    canvas.save(dst, "PNG")
    print(f"  🖼️ Avatar: {w}x{h} → {size}x{size}")

import sys
sys.path.append(WAV2LIP_DIR)

print("\n🔧 Patching inference.py for persistent VRAM caching...")
inf_path = os.path.join(WAV2LIP_DIR, "inference.py")
if os.path.exists(inf_path):
    with open(inf_path, "r") as f: code = f.read()
    if "_GLOBAL_MODEL = None" not in code:
        code = code.replace("def load_model(path):", "_GLOBAL_MODEL = None\n_GLOBAL_DETECTOR = None\ndef load_model(path):\n\tglobal _GLOBAL_MODEL\n\tif _GLOBAL_MODEL is not None: return _GLOBAL_MODEL")
        code = code.replace("return model.eval()", "_GLOBAL_MODEL = model.eval()\n\treturn _GLOBAL_MODEL")
        code = code.replace("face_detector = face_detection.FaceAlignment(face_detection.LandmarksType._2D, ", "global _GLOBAL_DETECTOR\n\tif _GLOBAL_DETECTOR is None:\n\t\t_GLOBAL_DETECTOR = face_detection.FaceAlignment(face_detection.LandmarksType._2D, ")
        code = code.replace("device=device)", "device=device)\n\tface_detector = _GLOBAL_DETECTOR")
        with open(inf_path, "w") as f: f.write(code)
        print("  ✅ Persistent cache patch applied.")

def run_wav2lip(scene_id, audio_path, avatar_path, output_mp4):
    """Run Wav2Lip inference using in-memory model."""
    temp_dir = os.path.join(WAV2LIP_DIR, "temp")
    for f in os.listdir(temp_dir):
        try: os.remove(os.path.join(temp_dir, f))
        except: pass
    
    prep = os.path.join(JOBS_DIR, f"{scene_id}_prep.png")
    prepare_avatar(avatar_path, prep)
    
    print(f"  🚀 Running In-Memory Wav2Lip Inference...")
    
    import sys
    # Mock sys.argv so inference.py's top-level parser doesn't crash on initial import
    old_argv = sys.argv
    sys.argv = [
        "inference.py", 
        "--checkpoint_path", os.path.join(WAV2LIP_DIR, "checkpoints", "wav2lip_gan.pth"),
        "--face", prep,
        "--audio", audio_path
    ]
    
    import inference
    import argparse
    
    sys.argv = old_argv
    
    # Fully populated namespace with all required Wav2Lip defaults
    args = argparse.Namespace(
        checkpoint_path=os.path.join(WAV2LIP_DIR, "checkpoints", "wav2lip_gan.pth"),
        face=prep,
        audio=audio_path,
        outfile=output_mp4,
        pads=[0, 10, 0, 0],
        resize_factor=1,
        nosmooth=True,
        fps=25.0,
        box=[-1, -1, -1, -1],
        static=False,
        img_size=96,
        face_det_batch_size=16,
        wav2lip_batch_size=128,
        crop=[0, -1, 0, -1],
        rotate=False
    )
    
    try:
        inference.args = args
        old_cwd = os.getcwd()
        os.chdir(WAV2LIP_DIR)
        inference.main()
        os.chdir(old_cwd)
    except Exception as e:
        print(f"  ❌ Inference Exception: {e}")
        traceback.print_exc()
        os.chdir(old_cwd)
        return False
    
    if os.path.exists(output_mp4) and os.path.getsize(output_mp4) > 1000:
        print(f"  ✅ Video: {os.path.getsize(output_mp4)} bytes")
        return True
    
    return False


@app.route("/health")
def health():
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    return jsonify({"status": "online", "gpu_name": gpu})

@app.route("/clear_cache")
def clear_cache():
    import torch, gc
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return jsonify({"status": "cleared"})

@app.route("/generate_lipsync", methods=["POST"])
def generate_lipsync():
    try:
        data = request.json
        scene_id = data.get("scene_id", "scene_01")
        
        # Save audio
        audio_path = os.path.join(JOBS_DIR, f"{scene_id}.wav")
        with open(audio_path, "wb") as f:
            f.write(base64.b64decode(data["audio_base64"]))
        print(f"  📁 Audio: {os.path.getsize(audio_path)} bytes")
        
        # Save avatar
        avatar_path = os.path.join(JOBS_DIR, f"{scene_id}_avatar.png")
        with open(avatar_path, "wb") as f:
            f.write(base64.b64decode(data["avatar_image_base64"]))
        print(f"  📁 Avatar: {os.path.getsize(avatar_path)} bytes")
        
        output_mp4 = os.path.join(JOBS_DIR, f"{scene_id}_sync.mp4")
        
        # Serialize GPU access
        print(f"🎬 {scene_id}: waiting for GPU...")
        with inference_lock:
            print(f"  🔓 GPU acquired")
            ok = run_wav2lip(scene_id, audio_path, avatar_path, output_mp4)
        
        if not ok:
            raise Exception(f"Wav2Lip produced no output for {scene_id}")
        
        with open(output_mp4, "rb") as f:
            video_b64 = base64.b64encode(f.read()).decode()
        
        print(f"  ✅ {scene_id} complete!")
        return jsonify({"status": "success", "lipsync_video_base64": video_b64})
        
    except Exception as e:
        print(f"🔥 Error: {traceback.format_exc()}")
        return jsonify({"status": "error", "error_message": str(e)}), 500

# ── Start Server ──
from pyngrok import ngrok

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("NGROK_AUTH_TOKEN") or NGROK_TOKEN
except:
    token = NGROK_TOKEN

if not token or "paste_your" in token:
    print("❌ Paste your ngrok token in NGROK_TOKEN variable!")
else:
    ngrok.set_auth_token(token)
    try:
        for t in ngrok.get_tunnels(): ngrok.disconnect(t.public_url)
    except: pass
    tunnel = ngrok.connect(5000)
    url = tunnel.public_url
    # Handle both string and NgrokTunnel object
    if hasattr(tunnel, 'public_url'):
        url = tunnel.public_url
    else:
        url = str(tunnel)
    # Extract clean URL if it contains extra text
    if 'https://' in url:
        import re
        match = re.search(r'(https://[^\s\"]+\.ngrok[^\s\"]*)', url)
        if match:
            url = match.group(1)
    print(f"\n🚀 PUBLIC URL: {url}")
    print(f"\n📋 Paste in backend/.env:")
    print(f"   CLOUD_RENDER_URL={url}")
    app.run(host="0.0.0.0", port=5000)